<a href="https://colab.research.google.com/github/GaganDeshbhandari/ML-major-project-notebook/blob/main/factllama_qlora_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("No GPU detected. Select a GPU runtime.")


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
# Cell 2
%pip install -q -U transformers datasets peft trl accelerate bitsandbytes sentencepiece huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 14.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
# Cell 3
import torch
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

print("\nInstalled packages:")
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

Installed packages:
Transformers: 5.19.0
Datasets: 5.1.0
PEFT: 0.21.2
TRL: 1.15.0
Accelerate: 1.15.0
BitsAndBytes: 0.50.2


In [ ]:
# Cell 4
from huggingface_hub import login
login()

In [ ]:
# Cell 5
from huggingface_hub import model_info

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

info = model_info(MODEL_ID)

print("Model:", info.id)
print("Model access verified.")

Model: meta-llama/Llama-3.2-3B-Instruct
Model access verified.


In [ ]:
# Cell 6
!apt-get -qq update
!apt-get -qq install -y unrar

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# Cell 7
!unrar x "/content/RAWFC.rar" "/content/RAWFC1/"


UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal

Cannot open /content/RAWFC.rar
No such file or directory
No files to extract


In [ ]:
# Cell 8
from pathlib import Path

RAWFC_DIR = Path("/content/RAWFC1")

files = [p for p in RAWFC_DIR.rglob("*") if p.is_file()]

print("Total extracted files:", len(files))

for file in files[:30]:
    print(file.relative_to(RAWFC_DIR))

Total extracted files: 0


In [ ]:
# Cell 9
from pathlib import Path

RAWFC_DIR = Path("/content/RAWFC1")

print("Directory exists:", RAWFC_DIR.exists())

all_files = [
    p for p in RAWFC_DIR.rglob("*")
    if p.is_file()
]

print("Total files:", len(all_files))
print("\nFirst 40 files:")

for file in all_files[:40]:
    print(file.relative_to(RAWFC_DIR))

Directory exists: True
Total files: 2013

First 40 files:
README.MD
val/211377.json
val/252257.json
val/247501.json
val/181491.json
val/224664.json
val/164582.json
val/197115.json
val/251416.json
val/151398.json
val/271059.json
val/204620.json
val/255951.json
val/164259.json
val/256347.json
val/164131.json
val/196641.json
val/244012.json
val/271433.json
val/105619.json
val/218204.json
val/239365.json
val/231098.json
val/208341.json
val/264365.json
val/253738.json
val/240933.json
val/215251.json
val/158697.json
val/197230.json
val/208855.json
val/155705.json
val/223050.json
val/255228.json
val/171734.json
val/180795.json
val/257110.json
val/267617.json
val/255183.json
val/247794.json


In [ ]:
# Cell 10
from collections import Counter
from pathlib import Path

RAWFC_DIR = Path("/content/RAWFC1")

# Count files in each top-level directory
for folder in sorted(RAWFC_DIR.iterdir()):
    if folder.is_dir():
        files = [p for p in folder.rglob("*") if p.is_file()]
        print(f"{folder.name}: {len(files)} files")

# Count file extensions
extensions = Counter(
    p.suffix.lower()
    for p in RAWFC_DIR.rglob("*")
    if p.is_file()
)
print("\nFile extensions:", extensions)

test: 200 files
train: 1612 files
val: 200 files

File extensions: Counter({'.json': 2012, '.md': 1})


In [ ]:
# Cell 11
import json

sample_file = next((RAWFC_DIR / "val").glob("*.json"))

with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("Sample file:", sample_file.name)
print("Top-level keys:", list(sample.keys()))

print("\nClaim:")
print(sample.get("claim"))

print("\nOriginal label:", sample.get("original_label"))
print("Label:", sample.get("label"))

print("\nNumber of reports:", len(sample.get("reports", [])))

for i, report in enumerate(sample.get("reports", [])[:2]):
    print(f"\nReport {i + 1}")
    print("Domain:", report.get("domain"))
    print("Evidence-marked sentences:")

    for sentence in report.get("tokenized", []):
        if sentence.get("is_evidence") == 1:
            print("-", sentence.get("sent")[:400])

Sample file: 211377.json
Top-level keys: ['event_id', 'claim', 'original_label', 'label', 'explain', 'reports']

Claim:
The toy company Hasbro acquired Death Row Records. 

Original label: Mostly True
Label: true

Number of reports: 23

Report 1
Domain: www.highsnobiety.com
Evidence-marked sentences:
- Death Row Records be found by Suge Knight and Dr. Dre and be one of the most powerful West Coast record label of the ’ 90 .

Report 2
Domain: thegrio.com
Evidence-marked sentences:
- Hasbro recently purchase Entertainment One Multimedia Company ( eOne ) for $ 4 billion .


In [ ]:
# Cell 12
from pathlib import Path
import json

RAWFC_DIR = Path("/content/RAWFC1")

for split in ["train", "val", "test"]:
    folder = RAWFC_DIR / split

    direct_json = list(folder.glob("*.json"))
    recursive_json = list(folder.rglob("*.json"))

    print(f"\n{split.upper()}")
    print("Directory exists:", folder.exists())
    print("Direct JSON files:", len(direct_json))
    print("Recursive JSON files:", len(recursive_json))
    print("First 5 files:", [p.name for p in recursive_json[:5]])

    invalid_json = []
    for file in recursive_json:
        try:
            with open(file, "r", encoding="utf-8") as f:
                json.load(f)
        except Exception as e:
            invalid_json.append((file.name, str(e)))

    print("Invalid JSON files:", len(invalid_json))
    if invalid_json:
        print("Examples:", invalid_json[:3])


TRAIN
Directory exists: True
Direct JSON files: 1612
Recursive JSON files: 1612
First 5 files: ['219247.json', '235226.json', '255236.json', '264968.json', '132844.json']
Invalid JSON files: 0

VAL
Directory exists: True
Direct JSON files: 200
Recursive JSON files: 200
First 5 files: ['211377.json', '252257.json', '247501.json', '181491.json', '224664.json']
Invalid JSON files: 0

TEST
Directory exists: True
Direct JSON files: 200
Recursive JSON files: 200
First 5 files: ['160739.json', '253488.json', '190891.json', '263478.json', '120604.json']
Invalid JSON files: 0


In [ ]:
# Cell 13
import json
from pathlib import Path
from collections import Counter

RAWFC_DIR = Path("/content/RAWFC1")

label_counts = Counter()
missing_fields = Counter()
evidence_counts = Counter()
examples = []

for split in ["train", "val", "test"]:
    files = sorted((RAWFC_DIR / split).glob("*.json"))
    split_labels = Counter()
    claims_with_evidence = 0

    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            record = json.load(f)

        for field in ["claim", "label", "reports"]:
            if not record.get(field):
                missing_fields[f"{split}:{field}"] += 1

        label = record.get("label")
        if label:
            label_counts[label] += 1
            split_labels[label] += 1

        evidence_sentences = []
        for report in record.get("reports", []):
            for item in report.get("tokenized", []):
                if item.get("is_evidence") == 1 and item.get("sent"):
                    evidence_sentences.append(item["sent"])

        evidence_counts[len(evidence_sentences)] += 1

        if evidence_sentences:
            claims_with_evidence += 1

        if split == "train" and len(examples) < 3:
            examples.append({
                "claim": record.get("claim"),
                "label": label,
                "evidence_count": len(evidence_sentences),
                "evidence_preview": evidence_sentences[:3]
            })

    print(f"\n{split.upper()}: {len(files)} records")
    print("Label distribution:", dict(split_labels))
    print("Claims with evidence:", claims_with_evidence)

print("\nALL-SPLIT LABEL DISTRIBUTION:", dict(label_counts))
print("Missing required fields:", dict(missing_fields))
print("\nSAMPLE TRAINING RECORDS:")

for example in examples:
    print(json.dumps(example, indent=2, ensure_ascii=False))


TRAIN: 1612 records
Label distribution: {'half': 537, 'true': 561, 'false': 514}
Claims with evidence: 1612

VAL: 200 records
Label distribution: {'half': 67, 'true': 67, 'false': 66}
Claims with evidence: 200

TEST: 200 records
Label distribution: {'half': 67, 'true': 67, 'false': 66}
Claims with evidence: 200

ALL-SPLIT LABEL DISTRIBUTION: {'half': 671, 'true': 695, 'false': 646}
Missing required fields: {}

SAMPLE TRAINING RECORDS:
{
  "claim": "Both Robert E. Lee and Jefferson Davis disavowed the Confederacy after the Civil War.",
  "label": "half",
  "evidence_count": 5,
  "evidence_preview": [
    "The Civil War memory historian David Blight note that although reconciliation be a “ noble and essential human impulse ” after a convulsive Civil War , reunion come at a devastate cost , a civil and political freedom for Black Americans become “ sacrificial offering on the altar of reunion.",
    "First , I be not a West Point Grad so maybe my opinion do n't belong here but I think Le

In [ ]:
# Cell 14
import json
from pathlib import Path
from collections import Counter

RAWFC_DIR = Path("/content/RAWFC1")
OUTPUT_DIR = Path("/content/factllama_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_LABELS = {"true", "half", "false"}

def extract_evidence(record):
    evidence = []
    seen = set()

    for report in record.get("reports", []):
        domain = report.get("domain", "unknown")

        for item in report.get("tokenized", []):
            sentence = item.get("sent", "").strip()

            if item.get("is_evidence") != 1 or not sentence:
                continue

            # Remove exact duplicate sentences within this claim.
            key = sentence.casefold()
            if key in seen:
                continue

            seen.add(key)
            evidence.append({
                "text": sentence,
                "domain": domain
            })

    return evidence

for split in ["train", "val", "test"]:
    input_dir = RAWFC_DIR / split
    output_file = OUTPUT_DIR / f"{split}.jsonl"

    written = 0
    skipped = 0
    evidence_counts = []
    label_counts = Counter()

    with output_file.open("w", encoding="utf-8") as out:
        for file in sorted(input_dir.glob("*.json")):
            with file.open("r", encoding="utf-8") as f:
                record = json.load(f)

            label = record.get("label")
            claim = record.get("claim", "").strip()
            evidence = extract_evidence(record)

            if label not in VALID_LABELS or not claim or not evidence:
                skipped += 1
                continue

            example = {
                "claim": claim,
                "evidence": evidence,
                "label": label
            }

            out.write(json.dumps(example, ensure_ascii=False) + "\n")

            written += 1
            label_counts[label] += 1
            evidence_counts.append(len(evidence))

    print(f"\n{split.upper()}")
    print("Saved records:", written)
    print("Skipped records:", skipped)
    print("Label counts:", dict(label_counts))
    print("Average evidence sentences:", round(
        sum(evidence_counts) / len(evidence_counts), 2
    ) if evidence_counts else 0)
    print("Maximum evidence sentences:", max(evidence_counts, default=0))
    print("Output:", output_file)

print("\nFiles created:")
for file in sorted(OUTPUT_DIR.glob("*.jsonl")):
    print(file.name, f"({file.stat().st_size / 1024:.1f} KB)")


TRAIN
Saved records: 1612
Skipped records: 0
Label counts: {'half': 537, 'true': 561, 'false': 514}
Average evidence sentences: 34.08
Maximum evidence sentences: 140
Output: /content/factllama_data/train.jsonl

VAL
Saved records: 200
Skipped records: 0
Label counts: {'half': 67, 'true': 67, 'false': 66}
Average evidence sentences: 32.23
Maximum evidence sentences: 111
Output: /content/factllama_data/val.jsonl

TEST
Saved records: 200
Skipped records: 0
Label counts: {'half': 67, 'true': 67, 'false': 66}
Average evidence sentences: 37.49
Maximum evidence sentences: 139
Output: /content/factllama_data/test.jsonl

Files created:
test.jsonl (1763.2 KB)
train.jsonl (12991.5 KB)
val.jsonl (1528.3 KB)


In [ ]:
# Cell 15
import json
from pathlib import Path
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
DATA_DIR = Path("/content/factllama_data")

# Load the tokenizer; this also checks model-repository access.
tok_loader = AutoTokenizer
tokenizer = tok_loader.from_pretrained(MODEL_ID)

SYSTEM_PROMPT = (
    "You are a fact-checking assistant. Evaluate the claim using the "
    "provided evidence. Return exactly one label: true, half, or false. "
    "Do not use outside knowledge."
)

def build_prompt(record, max_evidence=None):
    evidence = record["evidence"]
    if max_evidence is not None:
        evidence = evidence[:max_evidence]

    evidence_text = "\n".join(
        f"[{i + 1}] {item['text']}"
        for i, item in enumerate(evidence)
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Claim: {record['claim']}\n\n"
                f"Evidence:\n{evidence_text}\n\n"
                "Verdict:"
            )
        }
    ]

def count_tokens(messages):
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return len(tokenizer.encode(rendered, add_special_tokens=False))

with open(DATA_DIR / "train.jsonl", encoding="utf-8") as f:
    train_records = [json.loads(line) for line in f]

all_evidence_lengths = []
capped_lengths = []

for record in train_records:
    all_evidence_lengths.append(
        count_tokens(build_prompt(record))
    )
    capped_lengths.append(
        count_tokens(build_prompt(record, max_evidence=12))
    )

def summarize(name, values):
    values = sorted(values)
    n = len(values)
    print(f"\n{name}")
    print("Minimum:", values[0])
    print("Median:", values[n // 2])
    print("90th percentile:", values[int(n * 0.90)])
    print("Maximum:", values[-1])
    print("Prompts over 2048 tokens:", sum(x > 2048 for x in values))
    print("Prompts over 4096 tokens:", sum(x > 4096 for x in values))

print("Tokenizer loaded successfully:", MODEL_ID)
summarize("ALL EVIDENCE", all_evidence_lengths)
summarize("FIRST 12 EVIDENCE SENTENCES", capped_lengths)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully: meta-llama/Llama-3.2-3B-Instruct

ALL EVIDENCE
Minimum: 189
Median: 1317
90th percentile: 3135
Maximum: 5296
Prompts over 2048 tokens: 455
Prompts over 4096 tokens: 48

FIRST 12 EVIDENCE SENTENCES
Minimum: 189
Median: 595
90th percentile: 750
Maximum: 3082
Prompts over 2048 tokens: 1
Prompts over 4096 tokens: 0


In [ ]:
# Cell 16
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

assert torch.cuda.is_available(), "CUDA GPU is not available. Check Colab runtime type."

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nModel loaded and LoRA adapters configured.")
print("Allocated GPU memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

GPU: Tesla T4
GPU memory: 14.56 GB


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

Model loaded and LoRA adapters configured.
Allocated GPU memory: 2.91 GB


In [ ]:
# Cell 17
import json
from pathlib import Path
from datasets import Dataset

DATA_DIR = Path("/content/factllama_data")
MAX_LENGTH = 2048
MAX_EVIDENCE = 12
VALID_LABELS = {"true", "half", "false"}

SYSTEM_PROMPT = (
    "You are a fact-checking assistant. Evaluate the claim using only "
    "the provided evidence. Output exactly one verdict: true, half, or false."
)

def encode_record(record):
    label = record["label"].strip().lower()
    if label not in VALID_LABELS:
        raise ValueError(f"Unexpected label: {label}")

    evidence = record["evidence"][:MAX_EVIDENCE]
    evidence_text = "\n".join(
        f"[{i + 1}] {item['text']}" for i, item in enumerate(evidence)
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Claim: {record['claim']}\n\n"
                f"Evidence:\n{evidence_text}\n\n"
                "Verdict:"
            ),
        },
    ]

    # Prompt ends immediately before the assistant's verdict.
    prompt_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
    )

    # The desired assistant output is only the normalized verdict.
    target_ids = tokenizer.encode(
        label, add_special_tokens=False
    ) + [tokenizer.eos_token_id]

    if len(target_ids) >= MAX_LENGTH:
        raise ValueError("Target unexpectedly exceeds the context limit.")

    # Preserve the beginning of the prompt and always retain the target.
    max_prompt_length = MAX_LENGTH - len(target_ids)
    prompt_ids = prompt_ids[:max_prompt_length]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "length": len(input_ids),
        "verdict": label,
    }

encoded_datasets = {}

for split in ["train", "val", "test"]:
    path = DATA_DIR / f"{split}.jsonl"

    with path.open(encoding="utf-8") as f:
        records = [json.loads(line) for line in f]

    encoded = [encode_record(record) for record in records]
    dataset = Dataset.from_list(encoded)
    encoded_datasets[split] = dataset

    print(f"\n{split.upper()}")
    print("Examples:", len(dataset))
    print("Label distribution:", {
        label: sum(x["verdict"] == label for x in encoded)
        for label in sorted(VALID_LABELS)
    })
    print("Maximum encoded length:", max(x["length"] for x in encoded))
    print("Examples at context limit:",
          sum(x["length"] == MAX_LENGTH for x in encoded))

# Check that prompt tokens are masked and target tokens are supervised.
sample = encoded_datasets["train"][0]
supervised = [x for x in sample["labels"] if x != -100]

assert len(supervised) > 0
assert len(sample["input_ids"]) <= MAX_LENGTH
assert all(x == -100 for x in sample["labels"][:-
    len(supervised)])
assert tokenizer.decode(supervised, skip_special_tokens=True).strip() in VALID_LABELS

print("\nPreprocessing checks passed.")
print("Decoded sample target:",
      tokenizer.decode(supervised, skip_special_tokens=True))

In [ ]:
# Cell 18
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

assert torch.cuda.is_available(), "CUDA GPU is not available. Check Colab runtime type."

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nModel loaded and LoRA adapters configured.")
print("Allocated GPU memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")